# Regional Demand Archetypes Analysis
## Steel Distribution Market Segmentation

This notebook demonstrates the clustering analysis used to identify distinct types of metropolitan markets for steel distribution.

## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Paths
BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / 'features'
MODELS_DIR = BASE_DIR / 'models' / 'clustering'

print("Environment setup complete!")

In [ ]:
# Load data
df = pd.read_csv(DATA_DIR / 'msa_features_latest.csv')

print(f"Loaded {len(df)} metropolitan areas")
print(f"\nColumns: {df.columns.tolist()[:10]}...") 
df.head()

## 2. Exploratory Data Analysis

In [ ]:
# Summary statistics for key metrics
key_metrics = [
    'population',
    'manufacturing_emp',
    'construction_emp',
    'building_permits',
    'demand_intensity_score'
]

df[key_metrics].describe()

In [ ]:
# Distribution of demand intensity scores
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df['demand_intensity_score'], bins=20, color='steelblue', alpha=0.7, edgecolor='black')
axes[0].set_xlabel('Demand Intensity Score')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Demand Intensity Scores')
axes[0].grid(True, alpha=0.3)

# Box plot
axes[1].boxplot(df['demand_intensity_score'], vert=True)
axes[1].set_ylabel('Demand Intensity Score')
axes[1].set_title('Demand Intensity Box Plot')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Feature Correlation Analysis

In [ ]:
# Correlation matrix
corr_features = [
    'mfg_emp_per_capita',
    'constr_emp_per_capita',
    'permits_per_capita',
    'median_income',
    'population_growth',
    'demand_intensity_score'
]

correlation_matrix = df[corr_features].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=1)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 4. Clustering Analysis

In [ ]:
# Load cluster assignments (from clustering model)
try:
    df_clusters = pd.read_csv(MODELS_DIR / 'msa_cluster_assignments.csv')
    df = df.merge(df_clusters[['msa_name', 'cluster', 'cluster_name']], on='msa_name', how='left')
    print("Cluster assignments loaded successfully!")
    print(f"\nCluster distribution:")
    print(df['cluster_name'].value_counts())
except:
    print("Run clustering model first: python models/clustering/clustering_model.py")

In [ ]:
# Visualize cluster characteristics
if 'cluster_name' in df.columns:
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    axes = axes.flatten()
    
    metrics = [
        ('mfg_emp_per_capita', 'Manufacturing Employment per Capita'),
        ('constr_emp_per_capita', 'Construction Employment per Capita'),
        ('building_permits_growth', 'Building Permits Growth'),
        ('demand_intensity_score', 'Demand Intensity Score')
    ]
    
    for idx, (metric, title) in enumerate(metrics):
        ax = axes[idx]
        df.boxplot(column=metric, by='cluster_name', ax=ax)
        ax.set_title(title)
        ax.set_xlabel('Cluster')
        ax.set_ylabel(metric.replace('_', ' ').title())
        plt.sca(ax)
        plt.xticks(rotation=45, ha='right')
    
    plt.suptitle('Cluster Characteristics Comparison', fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()

## 5. Top Markets by Cluster

In [ ]:
# Top 5 MSAs in each cluster by demand intensity
if 'cluster_name' in df.columns:
    for cluster in df['cluster_name'].unique():
        if pd.notna(cluster):
            print(f"\n{'='*70}")
            print(f"Cluster: {cluster}")
            print('='*70)
            
            top_msas = df[df['cluster_name'] == cluster].nlargest(5, 'demand_intensity_score')
            
            display_df = top_msas[[
                'msa_name', 
                'state', 
                'demand_intensity_score',
                'population',
                'manufacturing_emp',
                'construction_emp'
            ]].reset_index(drop=True)
            
            print(display_df.to_string(index=False))

## 6. Geographic Distribution

In [ ]:
# Cluster distribution by census region
if 'cluster_name' in df.columns and 'census_region' in df.columns:
    cluster_region_ct = pd.crosstab(
        df['census_region'], 
        df['cluster_name'],
        normalize='index'
    ) * 100
    
    cluster_region_ct.plot(kind='bar', stacked=True, figsize=(12, 6), colormap='Set3')
    plt.title('Cluster Distribution by Census Region (%)', fontsize=14, weight='bold')
    plt.xlabel('Census Region')
    plt.ylabel('Percentage of MSAs')
    plt.legend(title='Cluster', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 7. Strategic Insights

### Key Takeaways:

1. **Market Segmentation:** Metropolitan areas fall into distinct archetypes based on manufacturing intensity, construction activity, and demographic trends.

2. **High-Priority Clusters:**
   - Manufacturing & Construction Hubs: Highest demand, requires large-scale facilities
   - Construction Boomtowns: Accelerating growth, medium-to-large facilities

3. **Geographic Patterns:** Sunbelt states (TX, AZ, FL) show clustering in high-growth segments, while Rustbelt markets (MI, IL) appear in mature/declining segments.

4. **Expansion Strategy:** Focus capital on high-growth clusters in underserved geographies.

## 8. Export Results

In [ ]:
# Export top opportunities for each cluster
if 'cluster_name' in df.columns:
    output_dir = BASE_DIR / 'reports'
    output_dir.mkdir(exist_ok=True)
    
    top_opportunities = df.nlargest(20, 'demand_intensity_score')[[
        'msa_name',
        'state',
        'cluster_name',
        'demand_intensity_score',
        'population',
        'manufacturing_intensity_index',
        'construction_momentum_index',
        'demographic_tailwind_index'
    ]]
    
    top_opportunities.to_csv(output_dir / 'top_20_opportunities.csv', index=False)
    print(f"Exported top opportunities to: {output_dir / 'top_20_opportunities.csv'}")
    print("\nTop 20 Opportunities:")
    display(top_opportunities)

## Next Steps

1. Review demand prediction models in `demand_forecast.ipynb`
2. Analyze momentum trends for timing decisions
3. Overlay competitive intelligence data
4. Develop ROI models for top candidates